In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

# 1. Chỉ định rõ model BERT để khớp với token [MASK]
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

# 2. Câu đầu vào
input_sentence = "Hanoi is the [MASK] of Vietnam."

# 3. Thực hiện dự đoán
predictions = mask_filler(input_sentence, top_k=5)

# 4. In kết quả
print(f"Mô hình đang sử dụng: {mask_filler.model.config._name_or_path}")
print(f"Câu gốc: {input_sentence}\n" + "-"*30)
for pred in predictions:
    print(f"Dự đoán: '{pred['token_str']}' | Độ tin cậy: {pred['score']:.4f}")
    print(f" -> Câu hoàn chỉnh: {pred['sequence']}")

In [ ]:
from transformers import pipeline

# 1. Tải pipeline text-generation với mô hình GPT-2
generator = pipeline("text-generation", model="gpt2")

# 2. Câu mồi
prompt = "The best thing about learning NLP is"

# 3. Sinh văn bản
# max_length: độ dài tối đa bao gồm cả câu mồi
generated_texts = generator(prompt, max_length=50, num_return_sequences=1, pad_token_id=50256)

# 4. In kết quả
print(f"Câu mồi: '{prompt}'\n" + "-"*30)
print("Văn bản được sinh ra:")
print(generated_texts[0]['generated_text'])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Khởi tạo Tokenizer và Model BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 2. Câu đầu vào
sentences = ["This is a sample sentence."]

# 3. Tokenize câu (chuyển chữ thành số)
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# 4. Đưa qua mô hình (không tính gradient để tiết kiệm RAM)
with torch.no_grad():
    outputs = model(**inputs)

# 5. Thực hiện Mean Pooling
# last_hidden_state chứa vector của từng từ một
last_hidden_state = outputs.last_hidden_state
attention_mask = inputs['attention_mask']

# Mở rộng mask để có cùng kích thước với hidden states
mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

# Tính tổng các vector nhân với mask (để bỏ qua padding) và chia trung bình
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
sentence_embedding = sum_embeddings / sum_mask

# 6. In kết quả
print("Kích thước vector biểu diễn (Embedding):", sentence_embedding.shape)
print("\n10 giá trị đầu tiên của vector:")
print(sentence_embedding[0][:10])